# 05 SAM-LAD Relation Probe

역할: SAM 기반 relation component model은 torch/SAM/NumPy 런타임 제약이 있으므로, 기존 orchestrator/dashboard와 분리해서 setup과 probe를 한 곳에서 관리한다.


## Cell Role: NumPy/Torch Runtime Guard

SAM-HQ는 torch를 import하므로, Colab의 NumPy 2.x 런타임에서는 먼저 NumPy 1.26.4로 맞춘 뒤 런타임을 재시작한다. 이 셀은 torch를 import하기 전에 가장 먼저 실행한다.


In [ ]:
from __future__ import annotations

import importlib.metadata
import subprocess
import sys

TARGET_NUMPY = '1.26.4'

try:
    numpy_version = importlib.metadata.version('numpy')
except importlib.metadata.PackageNotFoundError:
    numpy_version = '-'

print('numpy =', numpy_version)
if numpy_version == '-' or numpy_version.split('.', maxsplit=1)[0] != '1':
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--upgrade',
            '--force-reinstall',
            '--no-cache-dir',
            f'numpy=={TARGET_NUMPY}',
        ],
        check=True,
    )
    raise SystemExit(
        'Installed numpy==1.26.4. Restart the Colab runtime once, then rerun this notebook from the top.'
    )

print('numpy runtime is compatible with this torch/SAM stack')


## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM`을 최신 `main`으로 맞춘다. 로컬 실행 시에는 현재 checkout을 그대로 사용한다.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'

def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )

colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        local_rev = subprocess.check_output(
            ['git', '-C', str(colab_checkout), 'rev-parse', '--short', 'HEAD'],
            text=True,
        ).strip()
        remote_rev = subprocess.check_output(
            ['git', '-C', str(colab_checkout), 'rev-parse', '--short', 'origin/main'],
            text=True,
        ).strip()
        print(f'local HEAD={local_rev} origin/main={remote_rev}')
        if local_rev != remote_rev:
            print('sync src/relation from origin/main without touching local notebooks')
            subprocess.run(
                [
                    'git',
                    '-C',
                    str(colab_checkout),
                    'checkout',
                    'origin/main',
                    '--',
                    'experiments/validation/condition_shift_baseline/src/relation',
                ],
                check=True,
            )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and is_regram_root(p)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for source_root in (SRC_ROOT, SRC_ROOT / 'relation'):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Checkpoint And Data Readiness

SAM-HQ checkpoint, UniVAD가 제공하는 `segment_anything` package, position manifest, raw LOCO good images를 확인한다. checkpoint가 session에는 없고 Drive 공용 cache에 있으면 session 위치로 복사한다.


In [ ]:
CATEGORY = 'breakfast_box'
SEVERITY = 'high'
LIMIT = 30
MAX_COMPONENTS = 12
SAM_DEVICE = 'auto'
SAM_MIN_AREA_RATIO = 0.0005
SAM_SMALL_AREA_RATIO = 0.006
SAM_MAX_AREA_RATIO = 0.85
SAM_POINTS_PER_SIDE = 16  # 16 for smoke/quick run; 32 for fuller SAM-LAD masks
SAM_CROP_N_LAYERS = 0  # 0 is much faster; 1 is closer to SAM-LAD-style dense proposals
PROGRESS_EVERY = 1

POSITION_MANIFEST = REPO_ROOT / 'manifests' / 'query_position_shift.jsonl'
SAM_CHECKPOINT = REPO_ROOT / 'external' / 'UniVAD' / 'pretrained_ckpts' / 'sam_hq_vit_h.pth'
SAM_ROOT = REPO_ROOT / 'external' / 'UniVAD' / 'models'
DRIVE_MODEL_CHECKPOINT_ROOT = Path('/content/drive/MyDrive/ReGraM/model_checkpoints')
RAW_CATEGORY_GOOD = REPO_ROOT / 'data' / 'row' / 'mvtec_loco_anomaly_detection' / CATEGORY / 'test' / 'good'
OUTPUT_PATH = (
    EXP_ROOT
    / 'reports'
    / 'relation_probe'
    / f'{CATEGORY}_position_shift_{SEVERITY}_sam_lad_known_transform.json'
)

if Path('/content').exists() and not Path('/content/drive/MyDrive').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')

if not SAM_CHECKPOINT.exists():
    cached_checkpoint = DRIVE_MODEL_CHECKPOINT_ROOT / 'sam_hq_vit_h.pth'
    if cached_checkpoint.exists():
        SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['cp', '-n', str(cached_checkpoint), str(SAM_CHECKPOINT)], check=True)

required_paths = {
    'relation runner': SRC_ROOT / 'relation' / 'run_relation_probe.py',
    'position manifest': POSITION_MANIFEST,
    'raw LOCO test/good images': RAW_CATEGORY_GOOD,
    'SAM checkpoint': SAM_CHECKPOINT,
    'segment_anything package': SAM_ROOT / 'segment_anything',
}
missing_paths = {name: path for name, path in required_paths.items() if not path.exists()}
if missing_paths:
    raise FileNotFoundError(
        'Missing required SAM-LAD relation-probe path(s). '
        f'Run 01 setup cells first if external UniVAD/data are missing: {missing_paths}'
    )

display(pd.DataFrame([{
    'category': CATEGORY,
    'severity': SEVERITY,
    'component_model': 'sam_lad',
    'limit': LIMIT,
    'max_components': MAX_COMPONENTS,
    'sam_points_per_side': SAM_POINTS_PER_SIDE,
    'sam_crop_n_layers': SAM_CROP_N_LAYERS,
    'progress_every': PROGRESS_EVERY,
    'sam_checkpoint': str(SAM_CHECKPOINT),
    'sam_root': str(SAM_ROOT),
    'output': str(OUTPUT_PATH),
}]))


## Cell Role: Run Local Tests

실제 SAM inference 전에 SAM-LAD mask filtering/merge 로직과 relation geometry unit test를 먼저 확인한다.


In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'unittest',
        'experiments.validation.condition_shift_baseline.tests.test_sam_lad_components',
        'experiments.validation.condition_shift_baseline.tests.test_relation_geometry',
    ],
    cwd=REPO_ROOT,
    check=True,
)


## Cell Role: Run SAM-LAD Probe

SAM automatic masks로 component 후보를 만들고, 작은 mask는 object-level component로 병합한 뒤 known position transform relation score를 계산한다.


In [ ]:
cmd = [
    sys.executable,
    str(SRC_ROOT / 'relation' / 'run_relation_probe.py'),
    '--repo-root', str(REPO_ROOT),
    '--manifest', str(POSITION_MANIFEST),
    '--category', CATEGORY,
    '--severity', SEVERITY,
    '--limit', str(LIMIT),
    '--max-components', str(MAX_COMPONENTS),
    '--component-model', 'sam_lad',
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--sam-root', str(SAM_ROOT),
    '--sam-device', SAM_DEVICE,
    '--sam-min-area-ratio', str(SAM_MIN_AREA_RATIO),
    '--sam-small-area-ratio', str(SAM_SMALL_AREA_RATIO),
    '--sam-max-area-ratio', str(SAM_MAX_AREA_RATIO),
    '--sam-points-per-side', str(SAM_POINTS_PER_SIDE),
    '--sam-crop-n-layers', str(SAM_CROP_N_LAYERS),
    '--progress-every', str(PROGRESS_EVERY),
    '--output', str(OUTPUT_PATH),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## Cell Role: Inspect Scores

component 수, raw SAM mask 수, 작은 mask 병합 여부와 relation score median을 확인한다.


In [ ]:
summary = json.loads(OUTPUT_PATH.read_text())
metric_df = (
    pd.DataFrame([summary['metric_medians']])
    .T
    .reset_index()
    .rename(columns={'index': 'metric', 0: 'median'})
)
display(metric_df)

rows_df = pd.DataFrame(summary['rows'])
preview_cols = [
    'source_id',
    'component_model',
    'component_source',
    'component_count',
    'raw_mask_count',
    'merged_small_count',
    'component_sources',
    'component_note',
    's_abs',
    's_centered_norm',
    's_pair_norm',
]
display(rows_df[[col for col in preview_cols if col in rows_df.columns]].head(10))

if summary['skipped_count']:
    display(pd.DataFrame(summary['skipped']).head(10))
